In [3]:
# # ✅ 1. ENV SETUP
# import os
# from dotenv import load_dotenv

# load_dotenv(".env")

# FIGMA_TOKEN = os.getenv("FIGMA_TOKEN")
# FILE_ID_1 = os.getenv("FIGMA_DOCUMENT_ID1")
# FILE_ID_2 = os.getenv("FIGMA_DOCUMENT_ID2")
# SAS_URL = os.getenv("SAS_URL")

# pages_1 = ["Module 1", "Module 2", "Module 3", "Module 4"]
# pages_2 = ["Module 5", "Module 6", "Module 7", "Module 8"]

# required = ["FIGMA_TOKEN", "FILE_ID_1", "FILE_ID_2", "SAS_URL"]
# missing = [v for v in required if not globals()[v]]
# if missing:
#     raise EnvironmentError(f"❌ Missing env vars: {', '.join(missing)}")

# print("✅ Environment loaded.")

# # ✅ 2. UTILS
# import io
# import time
# from PIL import Image
# from contextlib import contextmanager

# # ---------- UTILS ----------
# def format_blob_name(folder: str, name: str) -> str:
#     return f"{folder}/{name.replace(':', '_').replace(' ', '_')}.webp"

# def format_path(name: str) -> str:
#     return name.replace(" ", "_").strip()

# @contextmanager
# def timed(label):
#     t0 = time.time()
#     yield
#     print(f"⏱️ {label} took {time.time() - t0:.2f}s")

# def convert_image_bytes_to_webp(image_bytes: bytes, min_height=1080, quality=95) -> bytes:
#     with Image.open(io.BytesIO(image_bytes)) as img:
#         img = img.convert("RGB")
#         if img.height < min_height:
#             scale_factor = min_height / img.height
#             new_width = int(img.width * scale_factor)
#             img = img.resize((new_width, min_height), Image.Resampling.LANCZOS)
#         buffer = io.BytesIO()
#         img.save(buffer, format="WEBP", quality=quality)
#         return buffer.getvalue()

# # ✅ 3. AZURE STORAGE
# import ssl
# from azure.storage.blob.aio import ContainerClient
# from azure.storage.blob import ContentSettings
# from azure.core.pipeline.transport import AioHttpTransport

# USE_INSECURE_SSL = False

# def get_container_client():
#     if USE_INSECURE_SSL:
#         ssl_context = ssl.create_default_context()
#         ssl_context.check_hostname = False
#         ssl_context.verify_mode = ssl.CERT_NONE
#         transport = AioHttpTransport(ssl_context=ssl_context)
#         return ContainerClient.from_container_url(SAS_URL, transport=transport)
#     return ContainerClient.from_container_url(SAS_URL)

# async def get_existing_blob_names(prefix: str = "") -> set[str]:
#     blob_names = set()
#     async with get_container_client() as client:
#         async for blob in client.list_blobs(name_starts_with=prefix):
#             blob_names.add(blob.name)
#     print(f"📦 {len(blob_names)} blobs found with prefix '{prefix}'")
#     return blob_names

# async def upload_image_blob(blob_name: str, data: bytes) -> str:
#     async with get_container_client() as client:
#         blob = client.get_blob_client(blob_name)
#         await blob.upload_blob(
#             data=data,
#             overwrite=True,
#             content_settings=ContentSettings(content_type="image/webp"),
#         )
#         return blob.url.split("?")[0]

# # ✅ 4. FIGMA FETCH + REFS
# import requests
# import json

# REFS_PATH = "figma_image_refs.json"

# def fetch_figma_file(file_id: str) -> dict:
#     res = requests.get(
#         f"https://api.figma.com/v1/files/{file_id}",
#         headers={"X-Figma-Token": FIGMA_TOKEN}
#     )
#     res.raise_for_status()
#     return res.json()

# # ---------- FIGMA IMAGE FETCH ----------
# def fetch_image_url_single(file_id: str, node_id: str, format="png", scale=1.0) -> str | None:
#     try:
#         res = requests.get(
#             f"https://api.figma.com/v1/images/{file_id}",
#             headers={"X-Figma-Token": FIGMA_TOKEN},
#             params={"ids": node_id, "format": format, "scale": scale},
#             timeout=30,
#         )
#         res.raise_for_status()
#         return res.json().get("images", {}).get(node_id)
#     except Exception as e:
#         print(f"❌ Error fetching URL for {node_id}: {e}")
#         return None

# def calculate_scale_to_target_height(node: dict, target_height: int = 1080) -> float:
#     """
#     Calculate the Figma image export scale needed to reach the desired height (e.g., 1080px).
#     Caps the scale to Figma’s maximum allowed value of 4.
#     """
#     try:
#         height = node.get("absoluteBoundingBox", {}).get("height", 1)
#         if height <= 0:
#             return 1.0
#         return min(round(target_height / height, 2), 4.0)
#     except Exception:
#         return 1.0

# def map_nodes_by_id(page: dict) -> dict[str, dict]:
#     """
#     Recursively walk a Figma page and return a dictionary mapping node IDs to their node objects.
#     Useful for looking up scale-relevant metadata like height.
#     """
#     node_map = {}

#     def walk(node):
#         if "id" in node:
#             node_map[node["id"]] = node
#         for child in node.get("children", []) or []:
#             walk(child)

#     walk(page)
#     return node_map


# def load_image_refs() -> dict[str, str]:
#     if os.path.exists(REFS_PATH):
#         with open(REFS_PATH, "r") as f:
#             return json.load(f)
#     return {}

# def save_image_refs(refs: dict[str, str]):
#     with open(REFS_PATH, "w") as f:
#         json.dump(refs, f, indent=2)

# # ✅ 5. FETCH IMAGE CONTENT
# import aiohttp

# async def fetch_image_content(url: str) -> bytes | None:
#     try:
#         async with aiohttp.ClientSession() as session:
#             async with session.get(url, timeout=30) as resp:
#                 if resp.status == 200:
#                     return await resp.read()
#                 print(f"⚠️ Failed to fetch {url} (status {resp.status})")
#     except Exception as e:
#         print(f"❌ Error fetching image: {e}")
#     return None

# # ✅ 6. IMAGE UPLOAD MAIN FUNCTION
# async def upload_figma_images(
#     file_id: str,
#     id_to_blobname: dict[str, str],
#     image_refs: dict[str, str],
#     existing_blobs: set[str],
#     cache_path: str,
#     overwrite: bool = False,
# ):
#     previous_refs = {}
#     if os.path.exists(cache_path):
#         try:
#             with open(cache_path, "r") as f:
#                 previous_refs = json.load(f)
#         except Exception as e:
#             print(f"❌ Failed to load cache: {e}")

#     to_upload = {
#         node_id: blob_name
#         for node_id, blob_name in id_to_blobname.items()
#         if overwrite
#         or blob_name not in existing_blobs
#         or image_refs.get(node_id) != previous_refs.get(node_id)
#     }

#     print(f"🧮 {len(id_to_blobname)} total | {len(to_upload)} to upload")

#     uploaded = 0
#     for i, (node_id, blob_name) in enumerate(to_upload.items(), 1):
#         url = fetch_image_url_single(file_id, node_id, scale=1.0)
#         if not url:
#             continue
#         image_bytes = await fetch_image_content(url)
#         if not image_bytes:
#             continue
#         webp = convert_image_bytes_to_webp(image_bytes)
#         await upload_image_blob(blob_name, webp)
#         print(f"✅ [{i}] Uploaded {blob_name}")
#         previous_refs[node_id] = image_refs[node_id]
#         try:
#             with open(cache_path, "w") as f:
#                 json.dump(previous_refs, f, indent=2)
#         except Exception as e:
#             print(f"❌ Failed to update cache after {blob_name}: {e}")
#         uploaded += 1

#     print(f"📊 Done | Uploaded: {uploaded} | Skipped: {len(id_to_blobname) - uploaded}")



# # ✅ 7. SCAN FIGMA PAGE
# import re

# def assign_parents(node: dict, parent: dict = None):
#     node["_parent"] = parent
#     for child in node.get("children", []) or []:
#         assign_parents(child, node)

# def extract_generic_image(node: dict, page_name: str) -> dict | None:
#     """
#     Extract image nodes by returning the RECTANGLE node that contains an IMAGE fill.
#     This ensures the exported image is not cropped by external frames.
#     """
#     if node.get("type") == "RECTANGLE":
#         for fill in node.get("fills", []):
#             if fill.get("type") == "IMAGE" and "imageRef" in fill:
#                 return {
#                     "node_id": node["id"],  # Export RECTANGLE directly
#                     "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
#                     "image_ref": fill["imageRef"],
#                     "type": "rectangle"
#                 }
#     return None


#     # Fallback: rectangle with image fill — export the rectangle node directly
#     if node.get("type") == "RECTANGLE":
#         fills = node.get("fills", [])
#         for fill in fills:
#             if fill.get("type") == "IMAGE" and "imageRef" in fill:
#                 return {
#                     "node_id": node["id"],  # ✅ Export this RECTANGLE node
#                     "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
#                     "image_ref": fill["imageRef"],
#                     "type": "rectangle"
#                 }

#     return None


# def scan_figma_page(page, structure_json=None):
#     scanned = {}
#     assign_parents(page)

#     def walk(node):
#         if node.get("type") == "RECTANGLE":
#             for fill in node.get("fills", []):
#                 if fill.get("type") == "IMAGE" and "imageRef" in fill:
#                     scanned[node["id"]] = {
#                         "node_id": node["id"],
#                         "blob_name": format_blob_name(f"Modules/{format_path(page['name'])}", node["id"]),
#                         "image_ref": fill["imageRef"],
#                         "type": "image"
#                     }
#         for child in node.get("children", []) or []:
#             walk(child)

#     walk(page)
#     print(f"🔍 {page['name']}: {len(scanned)} image rectangles found")
#     return scanned




# # ✅ 8. MAIN PIPELINE RUNNER
# import json

# structure_path = "../03_Outputs/SEA_Modules/en/module_structure.json"
# figma_path_1 = "../02_Inputs/figma_jsons/figma_document1.json"
# figma_path_2 = "../02_Inputs/figma_jsons/figma_document2.json"
# cache_path = "../03_Outputs/image_refs_cache.json"

# print("📦 Fetching existing Azure blobs...")
# existing_blobs = await get_existing_blob_names()

# print("🔄 Loading cached image references...")
# updated_refs = load_image_refs()

# with open(structure_path) as f:
#     structure_json = json.load(f)
# with open(figma_path_1) as f:
#     figma_data_1 = json.load(f)
# with open(figma_path_2) as f:
#     figma_data_2 = json.load(f)

# file_page_sets = [
#     (FILE_ID_1, figma_data_1["children"], pages_1),
#     (FILE_ID_2, figma_data_2["children"], pages_2)
# ]

# for file_id, figma_pages, page_names in file_page_sets:
#     for page_name in page_names:
#         page = next((p for p in figma_pages if p["name"] == page_name), None)
#         if not page:
#             print(f"⚠️ Page not found: {page_name}")
#             continue

#         print(f"\n📄 Processing page: {page_name}")
#         scanned = scan_figma_page(page, structure_json)
#         if not scanned:
#             continue

#         id_to_blobname = {
#             node_id: info["blob_name"].replace(" ", "_")
#             for node_id, info in scanned.items()
#         }

#         fetched_refs = {
#             node_id: info["image_ref"]
#             for node_id, info in scanned.items()
#             if "image_ref" in info
#         }

#         await upload_figma_images(
#             file_id=file_id,
#             id_to_blobname=id_to_blobname,
#             image_refs=fetched_refs,
#             existing_blobs=existing_blobs,
#             cache_path=cache_path,
#             overwrite=True,
#         )


# print("✅ Upload pipeline complete.")


✅ Environment loaded.
📦 Fetching existing Azure blobs...
📦 4604 blobs found with prefix ''
🔄 Loading cached image references...

📄 Processing page: Module 1
🔍 Module 1: 193 image rectangles found
🧮 193 total | 193 to upload
✅ [1] Uploaded Modules/Module_1/563_324.webp
✅ [2] Uploaded Modules/Module_1/609_345.webp
✅ [3] Uploaded Modules/Module_1/488_1680.webp
✅ [4] Uploaded Modules/Module_1/488_1685.webp
✅ [5] Uploaded Modules/Module_1/488_1690.webp
✅ [6] Uploaded Modules/Module_1/3661_206.webp
✅ [7] Uploaded Modules/Module_1/488_2078.webp
✅ [8] Uploaded Modules/Module_1/488_2070.webp
✅ [9] Uploaded Modules/Module_1/3623_76.webp
✅ [10] Uploaded Modules/Module_1/3623_67.webp
✅ [11] Uploaded Modules/Module_1/522_537.webp
✅ [12] Uploaded Modules/Module_1/3601_191.webp
✅ [13] Uploaded Modules/Module_1/3601_196.webp
✅ [14] Uploaded Modules/Module_1/3601_202.webp
✅ [15] Uploaded Modules/Module_1/488_1667.webp
✅ [16] Uploaded Modules/Module_1/613_471.webp
✅ [17] Uploaded Modules/Module_1/488_16

CancelledError: 

In [24]:
# ✅ ENV SETUP
import os
from dotenv import load_dotenv

load_dotenv(".env")

FIGMA_TOKEN = os.getenv("FIGMA_TOKEN")
FILE_ID_1 = os.getenv("FIGMA_DOCUMENT_ID1")
FILE_ID_2 = os.getenv("FIGMA_DOCUMENT_ID2")
SAS_URL = os.getenv("SAS_URL")

pages_1 = ["Module 1", "Module 2", "Module 3", "Module 4"]
pages_2 = ["Module 5", "Module 6", "Module 7", "Module 8"]

required = ["FIGMA_TOKEN", "FILE_ID_1", "FILE_ID_2", "SAS_URL"]
missing = [v for v in required if not globals()[v]]
if missing:
    raise EnvironmentError(f"❌ Missing env vars: {', '.join(missing)}")

print("✅ Environment loaded.")

# ✅ UTILS
import io
import time
from PIL import Image
from contextlib import contextmanager

def format_blob_name(folder: str, name: str) -> str:
    return f"{folder}/{name.replace(':', '_').replace(' ', '_')}.webp"

def format_path(name: str) -> str:
    return name.replace(" ", "_").strip()

@contextmanager
def timed(label):
    t0 = time.time()
    yield
    print(f"⏱️ {label} took {time.time() - t0:.2f}s")

def convert_image_bytes_to_webp(image_bytes: bytes, max_height: int = 1080, quality: int = 95) -> bytes:
    with Image.open(io.BytesIO(image_bytes)) as img:
        img = img.convert("RGB")

        # Calculate scale factor if height is above max_height
        if img.height > max_height:
            scale = max_height / img.height
            new_width = int(img.width * scale)
            new_height = max_height
            img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

        buffer = io.BytesIO()
        img.save(buffer, format="WEBP", quality=quality)
        return buffer.getvalue()


# ✅ AZURE STORAGE
import ssl
from azure.storage.blob.aio import ContainerClient
from azure.storage.blob import ContentSettings
from azure.core.pipeline.transport import AioHttpTransport

USE_INSECURE_SSL = False

def get_container_client():
    if USE_INSECURE_SSL:
        ssl_context = ssl.create_default_context()
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE
        transport = AioHttpTransport(ssl_context=ssl_context)
        return ContainerClient.from_container_url(SAS_URL, transport=transport)
    return ContainerClient.from_container_url(SAS_URL)

async def get_existing_blob_names(prefix: str = "") -> set[str]:
    blob_names = set()
    async with get_container_client() as client:
        async for blob in client.list_blobs(name_starts_with=prefix):
            blob_names.add(blob.name)
    print(f"📦 {len(blob_names)} blobs found with prefix '{prefix}'")
    return blob_names

async def upload_image_blob(blob_name: str, data: bytes) -> str:
    async with get_container_client() as client:
        blob = client.get_blob_client(blob_name)
        await blob.upload_blob(
            data=data,
            overwrite=True,
            content_settings=ContentSettings(content_type="image/webp"),
        )
        return blob.url.split("?")[0]

# ✅ FETCH IMAGE URLS
import requests
import json

REFS_PATH = "figma_image_refs.json"

def calculate_optimal_scale(canvas_height: float, target_height: int = 1080, max_scale: float = 4.0) -> float:
    """
    Calculate the optimal scale factor for Figma image export.

    Parameters:
    - canvas_height (float): The height of the image on the canvas.
    - target_height (int): The desired minimum height in pixels after scaling.
    - max_scale (float): The maximum scale Figma API allows.

    Returns:
    - float: A scale factor (rounded to 2 decimals) to use in the image export URL.
    """
    if not canvas_height or canvas_height <= 0:
        return 2.0  # sensible default if height is missing

    scale = target_height / canvas_height
    adjusted_scale=round(min(max_scale, max(1.0, scale)), 2)
    return adjusted_scale


def fetch_image_url_single(file_id: str, node_id: str, canvas_height: float, format="png") -> str | None:
    try:
        scale = calculate_optimal_scale(canvas_height)
        res = requests.get(
            f"https://api.figma.com/v1/images/{file_id}",
            headers={"X-Figma-Token": FIGMA_TOKEN},
            params={"ids": node_id, "format": format, "scale": scale},
            timeout=30,
        )
        res.raise_for_status()
        return res.json().get("images", {}).get(node_id)
    except Exception as e:
        print(f"❌ Error fetching URL for {node_id}: {e}")
        return None


def load_image_refs() -> dict[str, str]:
    if os.path.exists(REFS_PATH):
        with open(REFS_PATH, "r") as f:
            return json.load(f)
    return {}

def save_image_refs(refs: dict[str, str]):
    with open(REFS_PATH, "w") as f:
        json.dump(refs, f, indent=2)

# ✅ FETCH IMAGE CONTENT
import aiohttp

async def fetch_image_content(url: str) -> bytes | None:
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(url, timeout=30) as resp:
                if resp.status == 200:
                    return await resp.read()
                print(f"⚠️ Failed to fetch {url} (status {resp.status})")
    except Exception as e:
        print(f"❌ Error fetching image: {e}")
    return None

# ✅ SCAN FIGMA PAGE

from math import ceil

def extract_generic_image(node: dict, page_name: str) -> dict | None:
    if node.get("type") != "RECTANGLE":
        return None
    for fill in node.get("fills", []):
        if fill.get("type") == "IMAGE" and "imageRef" in fill:
            canvas_height = node.get("absoluteBoundingBox", {}).get("height", 0)
            return {
                "node_id": node["id"],
                "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
                "image_ref": fill["imageRef"],
                "canvas_height": node.get("absoluteBoundingBox", {}).get("height", 0),
                "type": "generic"
            }
    return None


def scan_figma_page(page):
    scanned = {}
    assign_parents(page)
    type_counts = {"generic": 0}

    def walk(node):
        if node.get("type") == "RECTANGLE":
            result = extract_generic_image(node, page["name"])
            if result:
                scanned[result["node_id"]] = result
                type_counts["generic"] += 1
        for child in node.get("children", []) or []:
            walk(child)

    walk(page)
    total = sum(type_counts.values())
    print(f"🔍 {page['name']}: {total} image rectangles found")
    return scanned


# ✅ UPLOAD
async def upload_figma_images(file_id, id_to_blobname, image_refs, existing_blobs, cache_path, overwrite=False):
    previous_refs = {}
    if os.path.exists(cache_path):
        try:
            with open(cache_path, "r") as f:
                previous_refs = json.load(f)
        except Exception as e:
            print(f"❌ Failed to load cache: {e}")

    to_upload = {
        node_id: blob_name
        for node_id, blob_name in id_to_blobname.items()
        if overwrite or blob_name not in existing_blobs or image_refs.get(node_id) != previous_refs.get(node_id)
    }

    print(f"🧮 {len(id_to_blobname)} total | {len(to_upload)} to upload")

    uploaded = 0
    for i, (node_id, blob_name) in enumerate(to_upload.items(), 1):
        canvas_height = scanned[node_id].get("canvas_height", 0)
        url = fetch_image_url_single(file_id, node_id, canvas_height)
        if not url:
            continue
        image_bytes = await fetch_image_content(url)
        if not image_bytes:
            continue
        webp = convert_image_bytes_to_webp(image_bytes)
        await upload_image_blob(blob_name, webp)
        print(f"✅ [{i}] Uploaded {blob_name}")
        previous_refs[node_id] = image_refs[node_id]
        try:
            with open(cache_path, "w") as f:
                json.dump(previous_refs, f, indent=2)
        except Exception as e:
            print(f"❌ Failed to update cache after {blob_name}: {e}")
        uploaded += 1

    print(f"📊 Done | Uploaded: {uploaded} | Skipped: {len(id_to_blobname) - uploaded}")


✅ Environment loaded.


In [ ]:
# ✅ 8. MAIN PIPELINE RUNNER
import json

structure_path = "../03_Outputs/SEA_Modules/en/module_structure.json"
figma_path_1 = "../02_Inputs/figma_jsons/figma_document1.json"
figma_path_2 = "../02_Inputs/figma_jsons/figma_document2.json"
cache_path = "../03_Outputs/image_refs_cache.json"

print("📦 Fetching existing Azure blobs...")
existing_blobs = await get_existing_blob_names()

print("🔄 Loading cached image references...")
updated_refs = load_image_refs()

with open(structure_path) as f:
    structure_json = json.load(f)
with open(figma_path_1) as f:
    figma_data_1 = json.load(f)
with open(figma_path_2) as f:
    figma_data_2 = json.load(f)

file_page_sets = [
    (FILE_ID_1, figma_data_1["children"], pages_1),
    (FILE_ID_2, figma_data_2["children"], pages_2)
]

for file_id, figma_pages, page_names in file_page_sets:
    for page_name in page_names:
        page = next((p for p in figma_pages if p["name"] == page_name), None)
        if not page:
            print(f"⚠️ Page not found: {page_name}")
            continue

        print(f"\n📄 Processing page: {page_name}")
        scanned = scan_figma_page(page)
        if not scanned:
            continue

        id_to_blobname = {
            node_id: info["blob_name"].replace(" ", "_")
            for node_id, info in scanned.items()
        }

        fetched_refs = {
            node_id: info["image_ref"]
            for node_id, info in scanned.items()
            if "image_ref" in info
        }

        await upload_figma_images(
            file_id=file_id,
            id_to_blobname=id_to_blobname,
            image_refs=fetched_refs,
            existing_blobs=existing_blobs,
            cache_path=cache_path,
            overwrite=True,
        )

print("✅ Upload pipeline complete.")

📦 Fetching existing Azure blobs...
📦 4604 blobs found with prefix ''
🔄 Loading cached image references...

📄 Processing page: Module 1
🔍 Module 1: 193 image rectangles found
🧮 193 total | 193 to upload
3.47
✅ [1] Uploaded Modules/Module_1/563_324.webp
3.69
✅ [2] Uploaded Modules/Module_1/609_345.webp
4.0
✅ [3] Uploaded Modules/Module_1/488_1680.webp
4.0
✅ [4] Uploaded Modules/Module_1/488_1685.webp
4.0
✅ [5] Uploaded Modules/Module_1/488_1690.webp
2.02
✅ [6] Uploaded Modules/Module_1/3661_206.webp
1.35
✅ [7] Uploaded Modules/Module_1/488_2078.webp
1.35
✅ [8] Uploaded Modules/Module_1/488_2070.webp
1.84
